# Python + NumPy 补课（CS231n 定制版）
> 适用对象：有 C++/Java 基础，目标 CS231n 作业。
> 方式：**先看代码、再填空、最后对答案**。每个练习都对应作业里会出现的真实用法。
> 全部完成 ≈ 3 小时。做错的题记到笔记里。

## 目录
1. Python 速览（C++/Java 对照）
2. NumPy：创建与属性
3. NumPy：索引与切片
4. NumPy：广播（最重要！）
5. NumPy：reshape / axis / 聚合
6. 实战：kNN 距离的三种实现


## 1. Python 速览（C++/Java 对照）
| 概念 | C++/Java | Python |
|---|---|---|
| 变量声明 | `int x = 3` | `x = 3`（动态类型，无需声明） |
| 数组/列表 | `int a[5]` / `ArrayList` | `a = [1,2,3]`（可混类型） |
| 循环 | `for(int i=0;i<n;i++)` | `for i in range(n):` |
| 函数 | `int f(int x) {...}` | `def f(x):`（无类型标注也行） |
| 打印 | `System.out.println` | `print(x)` |
| 判断 | `&&` / `\|\|` | `and` / `or` |
| 空值 | `null` | `None` |

### 必须掌握的 4 个 Python 特有写法（作业高频）


In [ ]:
# ① 列表推导式 —— 替代 90% 的 for+append
squares = [x*x for x in range(5)]
print(squares)  # [0, 1, 4, 9, 16]

# ② 切片 —— 比 C++ 方便一万倍
a = [0,1,2,3,4,5]
print(a[1:4])   # [1,2,3]  起点:终点(不含)
print(a[:3])    # [0,1,2]
print(a[::2])   # [0,2,4]  步长
print(a[::-1])  # [5,4,3,2,1,0]  反转

# ③ enumerate / zip —— 同时拿"下标"或"配对"
names = ['cat','dog','bird']
for i, name in enumerate(names):
    print(i, name)
for a_, b_ in zip([1,2,3], ['x','y','z']):
    print(a_, b_)

# ④ 字典 —— 就是 HashMap，但写法更短
d = {'a': 1, 'b': 2}
print(d.get('c', 0))  # 0  不存在时给默认值（作业常用）

In [ ]:
# 【练习 1】用列表推导式写：1~20 中的偶数，再乘以 3
# 期望输出: [6, 12, 18, 24, 30, 36, 42, 48, 54, 60]
evens = [x for x in range(1, 21) if x % 2 == 0]
result = [____ for ____ in ____]
print(result)

In [ ]:
# 【练习 2】切片：取出 a 的 [第2个, 第3个, 第4个] 元素
a = [10, 20, 30, 40, 50, 60]
part = a[____:____]
print(part)  # 期望 [20, 30, 40]

## 2. NumPy：创建与属性
NumPy 的 `ndarray` 就像 C++ 里"多维数组 + 数学运算"的合体。CS231n 里所有数据都是 `ndarray`：图片是 `(数量, 高, 宽, 通道)` 的四维数组。


In [ ]:
import numpy as np

# 创建
x = np.array([1, 2, 3])            # 一维
m = np.zeros((2, 3))               # 全0 (2行3列)
o = np.ones((2, 3))                # 全1
e = np.eye(3)                      # 单位阵
r = np.random.randn(2, 3)          # 标准正态随机

# 属性
print(x.shape)   # (3,)   形状 —— 最重要！
print(m.shape)   # (2, 3)
print(x.dtype)   # int64

# 形状像"嵌套列表"，但索引用逗号
print(m[0, 1])   # 第0行第1列
print(x[1])      # 一维就是普通下标

In [ ]:
# 【练习 3】创建：
# a) 3x4 的全0矩阵
# b) 长度为5、值为 [10,20,30,40,50] 的一维数组
# c) 输出它们的 shape
a = np.____((____, ____))
b = np.____([____])
print(a.shape, b.shape)   # 期望 (3, 4) (5,)

## 3. NumPy：索引与切片
和 Python 列表切片一样，但是**多维**的。作业里最常见的操作：
`X_train[mask]` —— 用布尔数组取子集（数据筛选）


In [ ]:
m = np.arange(12).reshape(3, 4)
# array([[ 0,  1,  2,  3],
#        [ 4,  5,  6,  7],
#        [ 8,  9, 10, 11]])

print(m[1])        # 第1行: [4 5 6 7]
print(m[:, 2])     # 第2列: [2 6 10]
print(m[0:2, 1:3]) # 前两行、中间两列

# 布尔索引（作业高频：筛数据）
arr = np.array([1, 5, 3, 8, 2])
print(arr[arr > 3])   # [5 8]

# 索引数组（作业高频：取某几行）
idx = np.array([0, 2])
print(arr[idx])       # [1 3]

In [ ]:
# 【练习 4】取出 m 的最后一行、最后一列
m = np.arange(12).reshape(3, 4)
last_row = m[____, :]
last_col = m[:, ____]
print(last_row, last_col)  # 期望 [8 9 10 11] / [3 7 11]

## 4. 广播（Broadcasting）—— 必须吃透！
规则：**形状不同的数组做运算时，NumPy 自动把小的"拉伸"到大的**。
C++/Java 里根本没有这个，所以最容易懵，但它是作业里最常用的省事大招。

例：`[1,2,3] + 10` → 10 被广播成 `[10,10,10]` → `[11,12,13]`


In [ ]:
x = np.array([1, 2, 3])
print(x + 10)        # [11 12 13]   标量广播
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m - np.array([1, 1, 1]))  # 每行都减 → 行向量沿行方向广播
# [[0 1 2]
#  [3 4 5]]

# 减"每列的平均" = 去均值（作业里归一化常用）
mean_per_col = m.mean(axis=0)   # [2.5 3.5 4.5]
print(m - mean_per_col)

In [ ]:
# 【练习 5】把 m 的每一行都除以 2
m = np.arange(12).reshape(3, 4)
result = m / ____
print(result[0])  # 期望 [0.  0.5 1.  1.5]

In [ ]:
# 【练习 6】给 x 加上一个列向量（考验广播方向）
x = np.array([[1, 2, 3],
              [4, 5, 6]])          # shape (2,3)
col = np.array([[10], [20]])       # shape (2,1)  → 会沿"列方向"广播
result = x + col
print(result)
# 期望:
# [[11 12 13]
#  [24 25 26]]

## 5. reshape / axis / 聚合
- `reshape`：改变形状但**不改变数据顺序**（作业里把图片拉平成向量 `X.reshape(N, -1)` 超常用）
- `axis`：轴。`axis=0` 沿"行方向"（竖着算，对每一列），`axis=1` 沿"列方向"（横着算，对每一行）


In [ ]:
x = np.arange(6)          # [0 1 2 3 4 5]
print(x.reshape(2, 3))  # [[0 1 2],[3 4 5]]
print(x.reshape(3, -1)) # -1 = 自动算，等价 (3,2)

m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m.sum(axis=0))   # [5 7 9]    每列之和
print(m.sum(axis=1))   # [6 15]     每行之和
print(m.max())         # 6          全局
print(np.argmax(m))    # 5          最大值的平铺索引
print(np.argsort([3, 1, 2]))  # [1 2 0]  排序后的下标（作业里取Top-K预测用）

In [ ]:
# 【练习 7】把 shape (4, 32, 32, 3) 的"图片数据"展平成 (4, 3072)
imgs = np.zeros((4, 32, 32, 3))   # 4张 32x32 彩色图
flat = imgs.reshape(4, -1)
print(flat.shape)  # 期望 (4, 3072)  ← CS231n 里干的事

# 【练习 8】算下面矩阵每列的最大值
m = np.array([[3, 8, 1],
              [6, 2, 9]])
col_max = m.max(axis=____)
print(col_max)  # 期望 [6 8 9]

## 6. 实战：kNN 距离的三种实现（Assignment 1 Q1 核心）
这是作业 1 里你**必须自己写**的部分：计算每个测试样本到每个训练样本的欧氏距离。
三种写法，从笨到聪明：
1. 双重循环（最慢，但先写对）
2. 单层循环 + 广播
3. 全向量化（公式展开，最快）

> 提示：`(a-b)^2 = a^2 + b^2 - 2ab`


In [ ]:
import numpy as np, time

# 造点假数据：500个训练样本、5个测试样本、每样本 32x32x3=3072 维
X_train = np.random.randn(500, 3072)
X_test  = np.random.randn(5, 3072)

# 写法1：双重循环（先写对）
def compute_distances_loops(X_test, X_train):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    for i in range(num_test):
        for j in range(num_train):
            # TODO: 填欧氏距离: sqrt(sum((X_test[i] - X_train[j])**2))
            dists[i, j] = ____
    return dists

# 写法2：单层循环 + 广播（理解后必写）
def compute_distances_one_loop(X_test, X_train):
    num_test = X_test.shape[0]
    dists = np.zeros((num_test, X_train.shape[0]))
    for i in range(num_test):
        # TODO: 利用广播一次算完一行：对每个训练样本算差的平方和再开方
        diff = X_train - X_test[i]          # (500, 3072)
        dists[i, :] = np.sqrt((diff ** 2).sum(axis=____))
    return dists

# 写法3：全向量化（作业要求最终达到的性能）
def compute_distances_no_loops(X_test, X_train):
    # 提示: (a-b)^2 = a^2 + b^2 - 2ab
    # sum(X_test^2, axis=1, keepdims=True) 是 (5,1)，广播!
    test_sq = np.sum(X_test ** 2, axis=1, keepdims=True)      # (5,1)
    train_sq = np.sum(X_train ** 2, axis=1)                    # (500,)
    cross = X_test @ X_train.T                                 # (5,500)  矩阵乘法!
    dists = np.sqrt(test_sq + train_sq - 2 * cross)
    return dists

# 填完上面 3 个 ____ 后运行下面代码验证三种写法结果一致
d1 = compute_distances_loops(X_test, X_train)
d2 = compute_distances_one_loop(X_test, X_train)
d3 = compute_distances_no_loops(X_test, X_train)
print('三种写法误差:', np.abs(d1 - d3).max(), np.abs(d2 - d3).max())
assert np.allclose(d1, d3, atol=1e-6), '写法1和3不一致!'
assert np.allclose(d2, d3, atol=1e-6), '写法2和3不一致!'
print('✅ 三种写法结果一致！')

# 计时对比
for name, fn in [('双重循环', compute_distances_loops),
                 ('单层+广播', compute_distances_one_loop),
                 ('全向量化', compute_distances_no_loops)]:
    t0 = time.time(); fn(X_test, X_train); t1 = time.time()
    print(f'{name}: {(t1-t0)*1000:.1f} ms')
# 你会看到 全向量化 比 双重循环 快几个数量级 —— 这就是学NumPy的意义

## ✅ 完成标准
- 所有练习输出与注释中的期望一致
- 三种 kNN 距离写法结果一致，且你**能说出**为什么向量化快
- 把本 notebook 完成后，复制到 `study-notes/notes/` 并提交 GitHub

## 答案速查（实在卡住再看）
- 练习1: `result = [x*3 for x in evens]`（或 `[x*3 for x in range(1,21) if x%2==0]`）
- 练习2: `a[1:4]`
- 练习3: `np.zeros((3,4))` / `np.array([10,20,30,40,50])`
- 练习4: `m[2, :]` / `m[:, 3]`
- 练习5: `result = m / 2`
- 练习6: 直接 `x + col`（广播自动处理，无需改动）
- 练习7: 无填空，直接看输出
- 练习8: `m.max(axis=0)`
- kNN: 循环版 `np.sqrt(np.sum((X_test[i]-X_train[j])**2))`；单循环 `axis=1`
